# Gemini method

In [4]:
import json
import os

def clean_sec_dataset(input_filepath: str, output_filepath: str, min_quality_score: float = 79.0):
    """
    Cleans a JSONL dataset by keeping only records with a quality score >= min_quality_score.
    Reads line-by-line for high performance and low memory usage.
    """
    kept_count = 0
    dropped_count = 0

    print(f"Starting cleaning process...\nThreshold: Quality Score >= {min_quality_score}")
    
    with open(input_filepath, 'r', encoding='utf-8') as infile, \
         open(output_filepath, 'w', encoding='utf-8') as outfile:
        
        for line_num, line in enumerate(infile, 1):
            if not line.strip():
                continue
            
            try:
                record = json.loads(line)
                
                # Extract quality score dynamically (handles different schema versions)
                score = extract_quality_score(record)
                
                # Filter condition
                if score >= min_quality_score:
                    # Write back to JSONL format
                    outfile.write(json.dumps(record) + '\n')
                    kept_count += 1
                else:
                    dropped_count += 1
                    
            except json.JSONDecodeError:
                print(f"Warning: Skipping line {line_num} due to invalid JSON format.")
                dropped_count += 1
                
    # Final Analytics
    total_processed = kept_count + dropped_count
    if total_processed > 0:
        retention_rate = (kept_count / total_processed) * 100
    else:
        retention_rate = 0.0

    print("\n--- Cleaning Complete ---")
    print(f"Total Processed: {total_processed}")
    print(f"Records Kept:    {kept_count} ({retention_rate:.1f}%)")
    print(f"Records Dropped: {dropped_count}")
    print(f"Clean file saved to: {output_filepath}")


def extract_quality_score(record: dict) -> float:
    """Helper function to safely locate the quality_score in nested JSON structures."""
    
    # 1. Check top-level metadata (v20 schema style)
    if 'metadata' in record and 'quality_score' in record['metadata']:
        return float(record['metadata']['quality_score'])
    
    # 2. Check inside instruction_pair if it's a dictionary (v19 schema style)
    if 'instruction_pair' in record and isinstance(record['instruction_pair'], dict):
        ip = record['instruction_pair']
        if 'metadata' in ip and 'quality_score' in ip['metadata']:
            return float(ip['metadata']['quality_score'])
            
    # 3. Check if instruction_pair is a JSON string (sometimes happens in raw generation)
    if 'instruction_pair' in record and isinstance(record['instruction_pair'], str):
        try:
            ip_dict = json.loads(record['instruction_pair'])
            if 'metadata' in ip_dict and 'quality_score' in ip_dict['metadata']:
                return float(ip_dict['metadata']['quality_score'])
        except json.JSONDecodeError:
            pass
            
    # Return 0 if not found, meaning it will be dropped safely
    return 0.0



In [5]:
# ==========================================
# HOW TO RUN IT
# ==========================================
if __name__ == "__main__":
    input_file = "01/sec_finetune_v19.jsonl"   # Your raw file
    output_file = "sec_finetune_v19-2_clean.jsonl"   # The new cleaned file
    
    clean_sec_dataset(
        input_filepath=input_file, 
        output_filepath=output_file, 
        min_quality_score=79.0  # Cuts everything UNDER 79
    )

Starting cleaning process...
Threshold: Quality Score >= 79.0

--- Cleaning Complete ---
Total Processed: 2948
Records Kept:    1223 (41.5%)
Records Dropped: 1725
Clean file saved to: sec_finetune_v19-2_clean.jsonl


## Gemini 2nd tern

In [1]:
import json
import pandas as pd
from collections import Counter

# Configuration
input_file = '01/sec_finetune_v19.jsonl'
output_file = 'sec_finetune_v19_clean_fixed.jsonl'
min_quality_score = 79.0

def robust_find_quality_score(obj):
    """
    Recursively search for quality_score. 
    Crucially, if it finds a string that looks like JSON, it parses it and searches inside it.
    """
    if isinstance(obj, dict):
        # Base case: we found it
        if 'quality_score' in obj:
            return float(obj['quality_score'])
        # Recursive step for dicts
        for v in obj.values():
            res = robust_find_quality_score(v)
            if res is not None:
                return res
                
    elif isinstance(obj, list):
        # Recursive step for lists
        for item in obj:
            res = robust_find_quality_score(item)
            if res is not None:
                return res
                
    elif isinstance(obj, str):
        # The Secret Weapon: Parse stringified JSON and search inside it
        text = obj.strip()
        if text.startswith('{') and text.endswith('}'):
            try:
                parsed = json.loads(text)
                return robust_find_quality_score(parsed)
            except json.JSONDecodeError:
                pass
                
    return None

# ==========================================
# Execution Phase
# ==========================================
scores = []
kept_count = 0
dropped_count = 0
missing_score_count = 0

print("Starting deep scan and cleaning...")

with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
    for line in infile:
        if not line.strip():
            continue
            
        try:
            record = json.loads(line)
            score = robust_find_quality_score(record)
            
            if score is not None:
                scores.append(score)
                # Cleaning Logic
                if score >= min_quality_score:
                    outfile.write(line)
                    kept_count += 1
                else:
                    dropped_count += 1
            else:
                missing_score_count += 1
                dropped_count += 1 # Drop records that have no score at all
        except Exception as e:
            missing_score_count += 1
            dropped_count += 1
            continue

# ==========================================
# Analytics Output
# ==========================================
if scores:
    total_found = len(scores)
    score_counts = Counter(scores)
    
    df_results = pd.DataFrame([
        {'Quality Score': k, 'Count': v, 'Proportion (%)': round((v / total_found) * 100, 2)}
        for k, v in score_counts.items()
    ])
    
    # Sorting by Score descending
    df_results = df_results.sort_values(by='Quality Score', ascending=False)
    
    print("\n" + "="*50)
    print(" TRUE SCORE DISTRIBUTION (Across all 13,000+ lines)")
    print("="*50)
    print(df_results.to_markdown(index=False))
    
    print("\n" + "="*50)
    print(" CLEANING RESULTS")
    print("="*50)
    print(f"Total Lines Scanned: {kept_count + dropped_count}")
    print(f"Records Kept (Score >= {min_quality_score}): {kept_count}")
    print(f"Records Dropped (Score < {min_quality_score}): {dropped_count - missing_score_count}")
    print(f"Records Dropped (NO SCORE FOUND/CORRUPT): {missing_score_count}")
    print(f"Clean file saved to: {output_file}")
    
else:
    print("CRITICAL ERROR: No quality scores found at all.")

Starting deep scan and cleaning...

 TRUE SCORE DISTRIBUTION (Across all 13,000+ lines)
|   Quality Score |   Count |   Proportion (%) |
|----------------:|--------:|-----------------:|
|           100   |     408 |            29.96 |
|            95   |      24 |             1.76 |
|            90   |     494 |            36.27 |
|            85   |      47 |             3.45 |
|            83.5 |       1 |             0.07 |
|            80   |     249 |            18.28 |
|            78.5 |       3 |             0.22 |
|            77   |       2 |             0.15 |
|            75.5 |       1 |             0.07 |
|            75   |      15 |             1.1  |
|            74   |       1 |             0.07 |
|            73.5 |       2 |             0.15 |
|            71   |       1 |             0.07 |
|            70   |      54 |             3.96 |
|            69   |       1 |             0.07 |
|            68.5 |       1 |             0.07 |
|            67   |       2 | 

In [2]:
import json, sys
from pathlib import Path

THRESHOLD = 79

INPUT  = "00_full_dataset.jsonl"
OUTPUT = Path(INPUT).stem + "claude_clean.jsonl"

print(f"Starting cleaning process... Threshold: Quality Score >= {THRESHOLD}")

kept = dropped = 0

with open(INPUT, "r", encoding="utf-8") as f_in, \
     open(OUTPUT, "w", encoding="utf-8") as f_out:

    for line in f_in:
        line = line.strip()
        if not line:
            continue

        record = json.loads(line)
        score  = record.get("instruction_pair", {}).get("metadata", {}).get("quality_score")

        if score is None or score >= THRESHOLD:
            f_out.write(line + "\n")
            kept += 1
        else:
            dropped += 1

total = kept + dropped
pct   = kept / total * 100 if total else 0

print(f"""
--- Cleaning Complete ---
Total Processed: {total:,}
Records Kept:    {kept:,} ({pct:.1f}%)
Records Dropped: {dropped:,}
Clean file saved to: {OUTPUT}""")

Starting cleaning process... Threshold: Quality Score >= 79


FileNotFoundError: [Errno 2] No such file or directory: '00_full_dataset.jsonl'

In [ ]:
import json

INPUT = "01/sec_finetune_v19.jsonl"
OUTPUT = "test_claude.jsonl"
THRESHOLD = 79.0

# Set to True if you want to keep records that haven't been scored by the auditor yet
# Set to False if you ONLY want records that explicitly scored 79+
KEEP_UNSCORED_RECORDS = False 

def robust_find_quality_score(obj):
    """Searches every possible nook and cranny for the score."""
    if isinstance(obj, dict):
        if 'quality_score' in obj: return float(obj['quality_score'])
        for v in obj.values():
            res = robust_find_quality_score(v)
            if res is not None: return res
    elif isinstance(obj, list):
        for item in obj:
            res = robust_find_quality_score(item)
            if res is not None: return res
    elif isinstance(obj, str):
        text = obj.strip()
        if text.startswith('{') and text.endswith('}'):
            try: return robust_find_quality_score(json.loads(text))
            except json.JSONDecodeError: pass
    return None

print(f"Starting Bulletproof Clean... Threshold: {THRESHOLD}")

kept = dropped_bad_score = dropped_no_score = 0

with open(INPUT, "r", encoding="utf-8") as f_in, open(OUTPUT, "w", encoding="utf-8") as f_out:
    for line in f_in:
        line = line.strip()
        if not line: continue

        record = json.loads(line)
        score = robust_find_quality_score(record)

        if score is not None:
            # We found a score! Is it good?
            if score >= THRESHOLD:
                f_out.write(line + "\n")
                kept += 1
            else:
                dropped_bad_score += 1
        else:
            # We COULD NOT find a score anywhere in this record
            if KEEP_UNSCORED_RECORDS:
                f_out.write(line + "\n")
                kept += 1
            else:
                dropped_no_score += 1

total = kept + dropped_bad_score + dropped_no_score

print("\n--- Cleaning Complete ---")
print(f"Total Processed: {total}")
print(f"Records Kept:    {kept}")
print(f"Dropped (Score < {THRESHOLD}): {dropped_bad_score}")
print(f"Dropped (No Score Found): {dropped_no_score}")

Starting Bulletproof Clean... Threshold: 79.0

--- Cleaning Complete ---
Total Processed: 13136
Records Kept:    5619
Dropped (Score < 79.0): 149
Dropped (No Score Found): 7368


# FOR ML project

In [7]:
import json
import pandas as pd
import numpy as np

def convert_jsonl_to_ml_csv(input_jsonl: str, output_csv: str, target_col: str = 'market_reaction'):
    """
    Transforms financial JSONL dataset into a flattened, cleaned, 
    and feature-engineered tabular CSV optimized for Machine Learning models.
    """
    flattened_records = []
    
    print(f"Reading and flattening records from {input_jsonl}...")
    
    with open(input_jsonl, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
                row_data = {}
                
                # 1. Base Unique Identifiers
                row_data['record_id'] = record.get('id', f'gen_{line_num}')
                row_data['record_type'] = record.get('record_type', 'unknown')
                
                # 2. Extract Top-Level Metadata / Root Metadata
                meta_block = record.get('meta', {})
                root_metadata = record.get('metadata', {})
                
                for k, v in meta_block.items():
                    if k != 'abnormal_return' and not isinstance(v, (dict, list)):
                        row_data[f'meta_{k}'] = v
                        
                for k, v in root_metadata.items():
                    if not isinstance(v, (dict, list)):
                        row_data[f'metadata_{k}'] = v

                # 3. Flatten Alternative Market Data (Abnormal Returns)
                abnormal_return = meta_block.get('abnormal_return', {})
                if abnormal_return:
                    for k, v in abnormal_return.items():
                        if not isinstance(v, (dict, list)):
                            row_data[f'market_{k}'] = v

                # 4. Extract & Flatten Financial Metrics 
                metrics_block = record.get('metrics', {})
                if metrics_block:
                    for k, v in metrics_block.items():
                        if isinstance(v, (int, float)):
                            row_data[f'metric_{k}'] = float(v)

                # 5. Extract Deep Core Signals (e.g., causal_density, cross_ref_validated)
                # This scans both standard keys and stringified 'instruction_pair' blocks
                signals_block = record.get('signals', {})
                if isinstance(signals_block, dict):
                    for k, v in signals_block.items():
                        if isinstance(v, (int, float, bool)):
                            row_data[f'signal_{k}'] = v
                            
                # Check inside 'instruction_pair' for embedded quantitative scores
                inst_pair = record.get('instruction_pair', '')
                if isinstance(inst_pair, str) and inst_pair.strip().startswith('{'):
                    try:
                        parsed_ip = json.loads(inst_pair)
                        # Extract any hidden scores or flags
                        for score_key in ['causal_density_score', 'quality_score']:
                            if score_key in parsed_ip:
                                row_data[f'extracted_{score_key}'] = parsed_ip[score_key]
                    except json.JSONDecodeError:
                        pass
                elif isinstance(inst_pair, dict):
                    # If it's already a dictionary, pull variables
                    row_data['extracted_causal_density'] = inst_pair.get('causal_density_score')

                flattened_records.append(row_data)

            except Exception as e:
                # Silently log errors to ensure robust execution
                continue

    # Convert the array of dictionaries into a Pandas Dataframe
    df = pd.DataFrame(flattened_records)
    print(f"Initial Flattening Complete. Discovered {len(df.columns)} unique dimensions.")

    # ==========================================
    # FEATURE ENGINEERING & ML SANITIZATION
    # ==========================================
    
    # Identify the chosen Target Label column dynamically
    target_candidates = [c for c in df.columns if target_col in c]
    if not target_candidates:
        # Fallback to an investment signal if market reaction is missing
        target_candidates = [c for c in df.columns if 'investment_signal' in c]
        
    if target_candidates:
        actual_target = target_candidates[0]
        print(f"Target column found and isolated: {actual_target}")
        
        # Strip string white spaces, encode non-null targets as Categorical integers
        df[actual_target] = df[actual_target].astype(str).str.strip().str.lower()
        df['ML_TARGET_LABEL'] = df[actual_target].astype('category').cat.codes
        
        # Keep track of class map for reference
        mapping = dict(enumerate(df[actual_target].astype('category').cat.categories))
        print(f"ML Class Labels Map encoded successfully: {mapping}")
    else:
        print("Warning: Direct target classification labels not identified. Adding dummy target.")
        df['ML_TARGET_LABEL'] = 0

    # Clean Up Columns: Drop text-heavy / narrative strings that cause ML shape distortion
    cols_to_drop = [
        c for c in df.columns 
        if any(word in c for word in ['company_name', 'company', 'date', 'no', 'text', 'rationale', 'sections', 'path', 'keywords'])
    ]
    df = df.drop(columns=cols_to_drop, errors='ignore')

    # Impute Missing Values across Numeric Matrix (Standard ML Protocol)
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    df[numeric_cols] = df[numeric_cols].fillna(0.0)

    # Encode Remaining Low-Cardinality Categorical variables (like form_type -> 10-K/10-Q)
    categorical_cols = df.select_dtypes(include=['object']).columns
    for col in categorical_cols:
        if df[col].nunique() < 15:  # Encode standard operational flags
            df[col] = df[col].astype('category').cat.codes
        else:
            # Drop high cardinality columns like raw IDs or CIK numbers to prevent overfitting
            df = df.drop(columns=[col], errors='ignore')

    # Final Export
    df.to_csv(output_csv, index=False)
    print(f"Success! Matrix structured and saved.\nFinal ML Matrix Dimensions: {df.shape[0]} samples x {df.shape[1]} clean features.")
    print(f"ML Dataset saved directly to: {output_csv}")

# ==========================================
# EXECUTION COMMAND
# ==========================================
if __name__ == "__main__":
    convert_jsonl_to_ml_csv(
        input_jsonl='00_full_dataset.jsonl',       # Input file
        output_csv='sec_ml_matrix_ready.csv',       # Output file for training
        target_col='market_reaction'               # Column we intend to predict
    )

Reading and flattening records from 00_full_dataset.jsonl...


Initial Flattening Complete. Discovered 90 unique dimensions.
Target column found and isolated: market_market_reaction
ML Class Labels Map encoded successfully: {0: 'negative', 1: 'neutral', 2: 'positive', 3: 'strong_negative', 4: 'strong_positive'}


/var/folders/fx/61s94vks26j1dz_f_1bkfbhw0000gn/T/ipykernel_46549/2024945208.py:122: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include=['object']).columns


Success! Matrix structured and saved.
Final ML Matrix Dimensions: 13136 samples x 79 clean features.
ML Dataset saved directly to: sec_ml_matrix_ready.csv


In [9]:
import pandas as pd
df = pd.read_csv("sec_ml_matrix_ready.csv")

df.isnull().sum().sort_values(ascending=False)


record_type                    0
metric_ebitda                  0
metric_return_on_assets_pct    0
metric_rd_ratio_pct            0
metric_sga_ratio_pct           0
                              ..
metric_income_tax              0
metric_pretax_income           0
metric_interest_expense        0
metric_operating_income        0
ML_TARGET_LABEL                0
Length: 79, dtype: int64